In [11]:
#  Import libraries and load data
import pandas as pd
import numpy as np
import seaborn as sn 
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler


#load data
df=pd.read_csv(r"C:\Users\HP\Desktop\credit-card-fraud-detection\data\creditcard.csv")

print("Data loaded successfully")
print("Shape:", df.shape)


Data loaded successfully
Shape: (284807, 31)


In [12]:
# look at first 5 rows
print("First 5 rows:")
df.head()

First 5 rows:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [13]:
# V1 to V28 are already scaled by PCA
# but Amount and Time are raw numbers — we need to scale them
# scaling means converting all values to same range
# so big numbers dont dominate the model


scaler=StandardScaler()
# scale Amount
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])

# scale Time
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

# create hour feature from Time
df['Hour'] = (df['Time'] / 3600) % 24

print("New columns added:")
print(df[['Amount', 'Amount_Scaled', 'Time', 'Time_Scaled', 'Hour']].head(10))



New columns added:
   Amount  Amount_Scaled  Time  Time_Scaled      Hour
0  149.62       0.244964   0.0    -1.996583  0.000000
1    2.69      -0.342475   0.0    -1.996583  0.000000
2  378.66       1.160686   1.0    -1.996562  0.000278
3  123.50       0.140534   1.0    -1.996562  0.000278
4   69.99      -0.073403   2.0    -1.996541  0.000556
5    3.67      -0.338556   2.0    -1.996541  0.000556
6    4.99      -0.333279   4.0    -1.996499  0.001111
7   40.80      -0.190107   7.0    -1.996436  0.001944
8   93.20       0.019392   7.0    -1.996436  0.001944
9    3.68      -0.338516   9.0    -1.996394  0.002500


In [14]:
# we no longer need raw Time and Amount

df.drop(['Time', 'Amount'], axis=1, inplace=True)

print("Columns after dropping:")
print(df.columns.tolist())
print("\nShape:", df.shape)

Columns after dropping:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'Amount_Scaled', 'Time_Scaled', 'Hour']

Shape: (284807, 32)


In [15]:
# X = everything the model uses to learn (features)
# y = what the model is trying to predict (fraud or not)

X = df.drop('Class', axis=1)
y = df['Class']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFraud cases in y:", y.sum())
print("Legit cases in y:", (y==0).sum())

X shape: (284807, 31)
y shape: (284807,)

Fraud cases in y: 492
Legit cases in y: 284315


In [18]:
from sklearn.model_selection import train_test_split

# split data into 80% training and 20% testing
# test_size=0.2 means 20% goes to testing
# random_state=42 means same split every time we run
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,
    random_state=42,
    stratify=y)


print("\nFraud in train:", y_train.sum())
print("Fraud in test:", y_test.sum())



Fraud in train: 394
Fraud in test: 98


In [19]:
from imblearn.over_sampling import SMOTE

# SMOTE creates fake but realistic fraud transactions
# so model can learn fraud patterns properly
# we only apply SMOTE on training data — never on test data

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:")
print("Fraud:", y_train.sum())
print("Legit:", (y_train==0).sum())

print("\nAfter SMOTE:")
print("Fraud:", y_train_smote.sum())
print("Legit:", (y_train_smote==0).sum())

Before SMOTE:
Fraud: 394
Legit: 227451

After SMOTE:
Fraud: 227451
Legit: 227451


In [20]:

# save the full processed dataframe
df.to_csv('../data/processed/processed_data.csv', index=False)

print("Processed data saved!")
print("Shape:", df.shape)
print("Location: data/processed/processed_data.csv")

Processed data saved!
Shape: (284807, 32)
Location: data/processed/processed_data.csv
